***

Preparing Workspace

***

California did not start collecting PM2.5 data until 1998 https://ww2.arb.ca.gov/resources/documents/fine-particulate-air-pollution-monitoring-program#:~:text=Federal%20Reference%20Method%20(FRM)%20Monitors,network%20in%20the%20late%201990s.

In [ ]:


## Importing packages ---

import numpy as np
import pandas as pd
import getpass
from pathlib import Path
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
import urllib.request, json
pd.set_option('display.max_columns', None)


## Setting file paths ---

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_raw = path_sp / 'Process Revamp' / 'Task 9. Collect new data' / 'EPA'
path_main = path_sp / 'Data'
path_prod = path_sp / 'Products'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'EPA'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'



## User defined functions ---

path_func = path_config0 / 'Functions.py'

with path_func.open("r") as f:
    exec(f.read())


## Setting the API key ---


# Obtain API Key from the following sources
# Air Quality Index (AQI): https://aqs.epa.gov/aqsweb/documents/data_api.html
# Clean Air Markets (CAM): https://www.epa.gov/power-sector/cam-api-portal#/
# Copy retrieved API key to .txt file for safe keeping

print("User input options:  'AQI', 'CAM', ...")
source = input('Which API source do you want to use? ')
file_api = path_config / 'api_key.txt'
exec(open(file_api).read())
api_key = dict_api[source]
print(api_key)


## Export setting ---

export=False



***

Importing

***

"CAMD collects data on emissions from fossil fuel-fired electric generating units (EGUs) with a nameplate capacity of over 25 MW"

It looks like the EIA also has power plant information, maybe more than the EPA

In [ ]:


if source == 'CAM':

    params = {
        'api_key': api_key,
        'year': 2023,
        # 'unitFuelType': 'Coal|Natural Gas',
        'page': 1,
        'perPage': 500,
        'stateCode':'CA'
    }
    
    url = "https://api.epa.gov/easey/facilities-mgmt/facilities/attributes"
    
    response = requests.get(url, params=params)
    data = json.loads(response.text)
    
    time.sleep(6)
    
    df_pg1 = pd.DataFrame(data)
    display(df_pg1)



In [ ]:
df_pg1.columns

In [ ]:
df_pg1['primaryFuelInfo'].value_counts()

In [ ]:
if source == 'CAM':
    params = {
        'api_key': api_key,
        'year': 2023,
        # 'unitFuelType': 'Coal|Natural Gas',
        'page': 2,
        'perPage': 500,
        'stateCode':'CA'
    }
    
    url = "https://api.epa.gov/easey/facilities-mgmt/facilities/attributes"
    
    response = requests.get(url, params=params)
    data = json.loads(response.text)
    
    time.sleep(6)
    
    df_pg2 = pd.DataFrame(data)
    df_pg2

In [ ]:
if source == 'CAM':
    params = {
        'api_key': api_key,
        'year': 2023,
        # 'unitFuelType': 'Coal|Natural Gas',
        'page': 2,
        'perPage': 500,
        'stateCode':'CA'
    }
    
    url = "https://api.epa.gov/easey/facilities-mgmt/facilities/attributes"
    
    response = requests.get(url, params=params)
    data = json.loads(response.text)
    
    time.sleep(6)
    
    df_pg2 = pd.DataFrame(data)
    df_pg2

In [ ]:

year_min = 1999
year_max = 2024

start_time = time.time()


if source == 'CAM':
    
    indicator_name = 'Health_6'
    params = {
        'api_key': api_key,
        'year': year_max,
        # 'unitFuelType': 'Coal|Natural Gas',
        'page': 1,
        'perPage': 500,
        'stateCode':'CA'
    }
    
    url = "https://api.epa.gov/easey/facilities-mgmt/facilities/attributes"
    
    response = requests.get(url, params=params)
    data = json.loads(response.text)
    
    time.sleep(6)
    
    df_epa = pd.DataFrame(data)


if source == 'AQI':

    indicator_name = 'Health_3'
    root_   = 'https://aqs.epa.gov/data/api/'
    email_  = 'jfontes@sacog.org'
    key_ = api_key
    data_   = 'dailyData'#'annualData'
    geo_    = 'CBSA'
    params  = ['88101', '44201']
    years   = sequence(year_min, year_max, 1) # 1980 earliest year, 1999 better for AQI
    geos    = ['40900']#, '49700'] # Sac and Yuba City
    
    list_df_geos = []
    
    for geo in geos:
        
        print(''); print('Importing MSA:', geo); print('')
        list_df_params = []
        
        for param in params:
            
            print(''); print('Importing parameter:', param)
            list_df_years = []
            
            for year in tqdm(years):
    
                time.sleep(6) # Sleeping for 6 seconds between requests to not upset the EPA overlords
                
                url_to_import = f"{root_}{data_}/by{geo_}?email={email_}&key={key_}&param={param}&bdate={year}0101&edate={year}1231&cbsa={geo}"
                
                with urllib.request.urlopen(url_to_import) as url:
                    dict_aqi = json.load(url)
                df_aqi = pd.DataFrame(dict_aqi['Data'])
                df_aqi['Year_Imported'] = year
                list_df_years.append(df_aqi)
                
            df_params = pd.concat(list_df_years)
            list_df_params.append(df_params)
            
        df_geo = pd.concat(list_df_params)
        list_df_geos.append(df_geo)
        print('')
    
    df_epa = pd.concat(list_df_geos)
    df_epa = df_epa.reset_index(drop=True)
    
    print("")
    print("Finished!!")
    print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")
    print('')
    

display(df_epa)



***

Exporting

***

In [ ]:

if export:
    geography='MSA'
    file_out = path_raw / f'{indicator_name}_{geography}_EPA_raw.csv'
    # df_epa.to_excel(file_out, index=False)
    df_epa.to_csv(file_out, index=False)

***

Testing

***

In [ ]:
import requests
import json
import sys
from datetime import datetime
from datetime import date
import os
import time

start_time = time.time()


# The bulk data api allows you to download prepackaged data sets. There are two endpoints for obtaining bulk data.
# The first is the /bulk-files endpoint which returns metadata about files. This metadata includes the path to the
# file.  The second is the /easey/bulk-files endpoint which along with the path, returns the actual file.

# Set your API key here
API_KEY = api_key
# S3 bucket url base + s3Path (in get request) = the full path to the files
BUCKET_URL_BASE = 'https://api.epa.gov/easey/bulk-files/'

parameters = {
    'api_key': API_KEY
}

# change this to the date you want to start downloading files from
# all files after this date and time will be downloaded
dateToday = date.today()
month, year = (dateToday.month-1, dateToday.year) if dateToday.month != 1 else (12, dateToday.year-1)
prevMonth = dateToday.replace(day=1, month=month, year=year)
timeOfLastDownload = datetime.fromisoformat(str(prevMonth)+"T00:00:00.000Z"[:-1] + '+00:00')

# executing get request
response = requests.get("https://api.epa.gov/easey/camd-services/bulk-files", params=parameters)

# printing the response error message if the response is not successful
print("Status code: "+str(response.status_code))
if (int(response.status_code) > 399):
    sys.exit("Error message: "+response.json()['error']['message'])

# converting the content from json format to a data frame
resjson = response.content.decode('utf8').replace("'", '"')
bulkFiles = json.loads(resjson)


## Calculate time amounted while requesting data
print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")
print('')


In [ ]:
bulkFiles

In [ ]:
df_bulk = pd.DataFrame(bulkFiles)
print(df_bulk.shape)
print(len(df_bulk['filename'].unique()))
df_bulk.head()

In [ ]:
# df_bulk.to_excel(os.path.join(path_config, 'bulkfiles.xlsx'),index=False)

In [ ]:
def extract_keys(dict):
    keys = ', '.join(list(dict.keys()))
    return keys

df_bulk['metadata_keys'] = df_bulk['metadata'].apply(extract_keys)

df_bulk

In [ ]:
df_bulk['metadata_keys'].value_counts()

In [ ]:

# filter by emissions files
emissionsFiles = [fileObj for fileObj in bulkFiles if (fileObj['metadata']['dataType']=="EDR")]

# filter by hourly virginia emissions files
# hourlyEmissionsFiles = [fileObj for fileObj in emissionsFiles if (fileObj['metadata']['dataSubType']=="Hourly")]
virginiaHourlyEmissionsFiles = [fileObj for fileObj in hourlyEmissionsFiles if ('stateCode' in fileObj['metadata'].keys() and fileObj['metadata']['stateCode'] == 'VA')]

# filter files since last download (timeOfLastDownload)
filesToDownload = [fileObj for fileObj in virginiaHourlyEmissionsFiles if datetime.fromisoformat(fileObj['lastUpdated'][:-1] + '+00:00') > timeOfLastDownload]
print('Number of files to download: '+str(len(filesToDownload)))

# print the size of all files to download
downloadMB = sum(int(fileObj['megaBytes']) for fileObj in filesToDownload)
print('Total size of files to download: '+str(downloadMB)+' MB')

# make a data folder if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

if len(filesToDownload) > 0:
    # loop through all files and download them
    for fileObj in filesToDownload:
        url = BUCKET_URL_BASE+fileObj['s3Path']
        print('Full path to file on S3: '+url)
        # download and save file
        response = requests.get(url)
        # save file to disk in the data folder
        with open('data/'+fileObj['filename'], 'wb') as f:
            f.write(response.content)
else:
    print('No files to download')

In [ ]:
list_dict = []

for dict in df_bulk['metadata'].values:
    keys = ', '.join(list(dict.keys()))
    list_dict.append(keys)

list_dict = list(set(list_dict))
list_dict

In [ ]:
for row in df_bulk.itertuples():
    print(row.Index, row.filename)


In [ ]:
# list_dict = []

# for row in df_bulk.itertuples():

#     dict = row.metadata
#     keys = ', '.join(list(dict.keys()))
    
#     df_bulk.loc[df_bulk['filename'] == row.filename, 'metadata_keys'] = keys

# df_bulk

In [ ]:
def extract_meta(dict):
    if dict is None:
        return pd.NA
    else:
        datatype    = dict['dataType'   ]
        year        = dict['year'       ]
        description = dict['description']
        stateCode   = dict['stateCode'  ]
        quarter     = dict['quarter'    ]

        return datatype, year , description, stateCode, quarter    

df_bulk = pd.DataFrame(bulkFiles)
df_bulk[['datatype', 'year', 'description', 'stateCode', 'quarter']] = df_bulk['metadata'].apply(lambda x: pd.Series(extract_meta(x)))

df_bulk